# Part 3 Transformer-Based Sentiment Analysis
Welcome to the state-of-the-art! In this notebook, we abandon traditional counting/statistics and sequence-level RNNs, and step into the world of **Transformers** using Hugging Face's `transformers` library.

We will:
1. Load a pretrained transformer (**BERT**).
2. Tokenize our text specifically for BERT.
3. Fine-tune BERT on our movie review dataset using the `Trainer` API.
4. Evaluate its performance and context understanding.

In [1]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Ensure GPU is used if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# We take a SMALL SUBSET of the data for this notebook. 
# Fine-tuning BERT on all 25,000 reviews takes several hours on a standard machine.
print("Loading a subset of the dataset for speed...")
train_df = pd.read_csv('../data/imdb_train.csv').sample(2000, random_state=42)
test_df = pd.read_csv('../data/imdb_test.csv').sample(500, random_state=42)

train_df['label'] = train_df['sentiment'].map({"Negative": 0, "Positive": 1})
test_df['label'] = test_df['sentiment'].map({"Negative": 0, "Positive": 1})

# Convert Pandas DataFrames into Hugging Face Datasets
train_dataset = Dataset.from_pandas(train_df[['text', 'label']])
test_dataset = Dataset.from_pandas(test_df[['text', 'label']])

Using device: cpu
Loading a subset of the dataset for speed...


## Step 1 & 2 Load Pretrained Tokenizer & Model
BERT understands text completely differently than TF-IDF. It breaks words into "sub-words" (e.g., "playing" -> "play", "##ing") and maps them to highly contextual mathematical vectors.

We use `bert-base-uncased` which means all text is converted to lowercase.

In [2]:
model_name = "bert-base-uncased"

print(f"Loading tokenizer: {model_name}")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading model architecture...")
# We specify num_labels=2 because we have Positive (1) and Negative (0)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
model.to(device)

def tokenize_function(examples):
    # Padding and truncation ensure all inputs are the exact same length (512 tokens max for BERT)
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=256)

print("Tokenizing training data...")
tokenized_train = train_dataset.map(tokenize_function, batched=True)

print("Tokenizing testing data...")
tokenized_test = test_dataset.map(tokenize_function, batched=True)

Loading tokenizer: bert-base-uncased
Loading model architecture...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Tokenizing training data...


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing testing data...


Map:   0%|          | 0/500 [00:00<?, ? examples/s]

## Step 3 Fine-Tuning with Trainer API
Hugging Face provides an incredible class called `Trainer` that handles all the complex PyTorch training loops for us.
We just need to define how we want it to evaluate (e.g., compute accuracy) and define our training hyperparameters.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='binary')
    return {
        'accuracy': acc,
        'f1': f1,
        'precision': precision,
        'recall': recall
    }

# Define training arguments
training_args = TrainingArguments(
    output_dir='../models/bert_sentiment',
    num_train_epochs=2,              # Train for 2 passes over the data
    per_device_train_batch_size=8,   # Small batch size to avoid running out of memory
    per_device_eval_batch_size=16,
    evaluation_strategy="epoch",     # Evaluate at the end of each epoch
    save_strategy="epoch",
    logging_dir='./logs',
    learning_rate=2e-5,              # Tiny learning rate because BERT is already pre-trained
    weight_decay=0.01,
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics
)

print("Starting Fine-Tuning (This will take time depending on your hardware!)...")
trainer.train()

## Step 4 Evaluate Results
Now we see how well BERT performs on the test set.

In [4]:
print("Evaluating on test set...")
results = trainer.evaluate()
print("\n--- BERT Results ---")
for key, value in results.items():
    print(f"{key}: {value}")

# Save the final model
trainer.save_model('../models/bert_sentiment_final')
tokenizer.save_pretrained('../models/bert_sentiment_final')
print("Model saved successfully!")

Evaluating on test set...


  0%|          | 0/32 [00:00<?, ?it/s]

{'eval_loss': 0.4772288501262665, 'eval_accuracy': 0.832, 'eval_f1': 0.8372093023255814, 'eval_precision': 0.7659574468085106, 'eval_recall': 0.9230769230769231, 'eval_runtime': 51.6595, 'eval_samples_per_second': 9.679, 'eval_steps_per_second': 0.619, 'epoch': 0.15}

--- BERT Results ---
eval_loss: 0.4772288501262665
eval_accuracy: 0.832
eval_f1: 0.8372093023255814
eval_precision: 0.7659574468085106
eval_recall: 0.9230769230769231
eval_runtime: 51.6595
eval_samples_per_second: 9.679
eval_steps_per_second: 0.619
epoch: 0.148
Model saved successfully!


## Context Understanding & Sarcasm (Bonus Comparison)
Traditional models struggle with sarcasm because they only look at individual words. BERT looks at the entire sentence context bidirectionally. Let's test it on a tricky sentence!

In [ ]:
from transformers import pipeline

# Load our newly trained model into an easy-to-use pipeline
sentiment_pipeline = pipeline("sentiment-analysis", model='../models/bert_sentiment_final', tokenizer='../models/bert_sentiment_final')

tricky_sentences = [
    "I absolutely loved wasting two hours of my life on this garbage.", # Sarcasm
    "The movie was not terrible, I actually quite liked it.",          # Negation
    "A masterpiece of terrible writing."                               # Contradictory
]

print("--- BERT Inference on Tricky Sentences ---")
for sentence in tricky_sentences:
    result = sentiment_pipeline(sentence)[0]
    # label_1 usually means positive in our mapping, label_0 means negative.
    # The pipeline outputs LABEL_0 or LABEL_1.
    label_name = "Positive" if result['label'] == 'LABEL_1' else "Negative"
    print(f"Sentence: '{sentence}'")
    print(f"Prediction: {label_name} (Confidence: {result['score']:.4f})\n")